# 📖 PDF → audiobook mp3 bằng giọng clone (OmniVoice) trên Colab

Upload **một file PDF** + **clip giọng mẫu**. Notebook tách câu, render bằng giọng clone trên GPU T4, rồi **ghép thành file mp3** (mỗi chương một file, hoặc cả cuốn một file) lưu trên Google Drive để bạn tải về.

Render được lưu **từng đợt lên Drive**: runtime chết giữa chừng thì chạy lại, nó tiếp tục từ chỗ dừng.

**Luồng:** upload → tách câu & kiểm tra bằng mắt → gán sắc thái theo ngữ cảnh → *benchmark 30 câu* → render hết → báo cáo → ghép mp3 → tải về.

## 0. Kiểm tra GPU (phải thấy Tesla T4)

In [ ]:
!nvidia-smi

## 1. Cài đặt

In [ ]:
!pip install -q omnivoice soundfile pymupdf anthropic
print('Xong.')

## 2a. Mount Drive + upload PDF

Chạy ô dưới, bấm nút và chọn **một file PDF**. Notebook copy nó vào Drive.
Runtime chết giữa chừng: điền `BOOK` đúng tên thư mục trên Drive thì ô này bỏ qua bước upload và nạp lại sách đã lưu.

In [ ]:
from google.colab import drive, files
import re, shutil, unicodedata
from pathlib import Path

drive.mount('/content/drive')

BOOK = ''   # trống = sách mới; điền tên thư mục (vd 'dac_nhan_tam') để tiếp tục sách đã upload
BASE = Path('/content/drive/MyDrive/omnivoice_render')

def slugify(name):
    s = Path(name).stem.replace('đ', 'd').replace('Đ', 'D')
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-zA-Z0-9]+', '_', s).strip('_').lower() or 'book'

REF_AUDIO = None
if BOOK and (BASE/BOOK/'source.pdf').exists():
    ROOT = BASE/BOOK
    PDF = str(ROOT/'source.pdf')
    refs = sorted(ROOT.glob('ref.*'))
    REF_AUDIO = str(refs[0]) if refs else None
    print('Tiếp tục sách:', BOOK, '| giọng mẫu đã lưu:', Path(REF_AUDIO).name if REF_AUDIO else 'chưa có')
else:
    if BOOK:
        print(f"Không thấy '{BOOK}' trên Drive → coi như sách mới.")
    up = files.upload()
    pdf_name = next((n for n in up if n.lower().endswith('.pdf')), None)
    assert pdf_name, 'Bạn chưa chọn file .pdf. Chạy lại ô này.'
    BOOK = slugify(pdf_name)
    ROOT = BASE/BOOK
    ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copy(pdf_name, ROOT/'source.pdf')
    PDF = str(ROOT/'source.pdf')

print('Sách:', BOOK, '| thư mục Drive:', ROOT)

## 2b. Upload giọng mẫu

Chạy ô dưới, bấm nút và chọn **một clip giọng** (wav/mp3/m4a/flac/ogg).
Sách đã có giọng mẫu lưu trên Drive thì ô này tự bỏ qua. Muốn đổi sang giọng khác: đặt `REPLACE_VOICE = True` (và **nhớ đổi `VOICE_ID`** ở mục 5, để cache không lẫn giọng cũ).

In [ ]:
REPLACE_VOICE = False   # True = upload giọng mẫu mới cho sách này

assert 'ROOT' in globals(), 'Hãy chạy ô 2a (upload PDF) trước.'
if REF_AUDIO and not REPLACE_VOICE:
    print('Dùng giọng mẫu đã lưu:', Path(REF_AUDIO).name)
else:
    up = files.upload()
    ref_name = next((n for n in up if n.lower().endswith(('.wav', '.mp3', '.m4a', '.flac', '.ogg'))), None)
    assert ref_name, 'Bạn chưa chọn clip giọng (.wav/.mp3/.m4a/.flac/.ogg). Chạy lại ô này.'
    for old in ROOT.glob('ref.*'):
        old.unlink()
    ref_dst = ROOT/('ref' + Path(ref_name).suffix)
    shutil.copy(ref_name, ref_dst)
    REF_AUDIO = str(ref_dst)
    print('Đã lưu giọng mẫu:', ref_dst.name)

## 3. Tải / cập nhật code từ GitHub

Phần xử lý nằm trong repo [nguyenkha283/kindle-voice](https://github.com/nguyenkha283/kindle-voice):
- `backend/text_norm.py`: chuẩn hoá chữ trước khi đọc (chữ VIẾT HOA, số La Mã, ký hiệu lạ);
- `colab/pdf_units.py`: PDF → danh sách câu;
- `colab/sac_thai.py`: sắc thái giọng đọc (clip mẫu + tốc độ + khoảng nghỉ, và cách gán sắc thái).

**Khi có bản sửa mới, không cần mở notebook mới:** chạy lại ô dưới để kéo code mới nhất, rồi chạy lại các mục bị ảnh hưởng (thường là mục 4 và 4b). Dữ liệu trên Drive (câu đã render, điểm sắc thái, mp3) vẫn giữ nguyên.

Muốn chỉnh tốc độ / khoảng nghỉ của một sắc thái thì sửa ngay trong notebook, sau ô này, ví dụ `sac_thai.REGISTERS['tinh_cam']['speed'] = 0.95`.

In [ ]:
import subprocess, sys, importlib
from pathlib import Path

REPO = 'https://github.com/nguyenkha283/kindle-voice.git'
REPO_BRANCH = 'claude/eloquent-noether-h8i4wo'   # đổi thành 'main' khi bản này đã được merge
CODE = Path('/content/kindle-voice')

def git(*args):
    subprocess.run(['git', *args], check=True, capture_output=True, text=True)

if (CODE/'.git').exists():
    git('-C', str(CODE), 'fetch', '-q', '--depth', '1', 'origin', REPO_BRANCH)
    git('-C', str(CODE), 'reset', '-q', '--hard', 'FETCH_HEAD')
else:
    git('clone', '-q', '--depth', '1', '-b', REPO_BRANCH, REPO, str(CODE))
for d in ('backend', 'colab'):
    if str(CODE/d) not in sys.path:
        sys.path.insert(0, str(CODE/d))

import text_norm, pdf_units, sac_thai
for m in (text_norm, pdf_units, sac_thai):   # nạp lại bản mới nếu đã import từ trước
    importlib.reload(m)
log = subprocess.run(['git', '-C', str(CODE), 'log', '-1', '--format=%h %cd %s', '--date=format:%d/%m %H:%M'],
                     capture_output=True, text=True).stdout.strip()
print('Code:', log)

## 4. PDF → danh sách câu (xem kỹ trước khi render!)

**Đọc lướt phần đầu và phần giữa sách.** Đây là bước duy nhất ngăn bạn render 4 tiếng GPU cho một đống rác. Cần để ý:
- chú thích cuối trang, số trang, tiêu đề lặp bị đọc lẫn vào câu;
- sách hai cột hoặc có bảng thì thứ tự đọc có thể sai;
- chương: nếu PDF không có mục lục thì chia tạm mỗi 20 trang.

In [ ]:
import json
from collections import Counter
import pdf_units

OCR = False       # True nếu PDF là bản scan (chạy ô cài tesseract ngay bên dưới trước)
MAX_CHARS = 220   # câu dài hơn bị cắt ở dấu ; : , gần nhất

res = pdf_units.build_units(PDF, max_chars=MAX_CHARS, ocr=OCR)
units = res['units']
print(res['stats'])
for w in res['warnings']:
    print('⚠️', w)

cc = Counter(u['ch'] for u in units)
print(f"\nChương (nguồn: {res['stats']['chapter_source']}):")
for c in res['chapters'][:40]:
    print(f"  {c['id']:>3} tr.{c['start_page']:<4} {cc[c['id']]:>5} câu  {c['title'][:60]}")
if len(res['chapters']) > 40:
    print('  ...')

def show(title, rows):
    print(f'\n--- {title}')
    for u in rows:
        print(f"[{u['i']} | tr.{u['page']}] {u['text'][:160]}")

mid = len(units) // 2
show('8 câu đầu', units[:8])
show('8 câu ở giữa sách', units[mid:mid + 8])
show('3 câu dài nhất', sorted(units, key=lambda u: -len(u['text']))[:3])

changed = [u for u in units if u['say'] != u['text']]
print(f'\n--- {len(changed)} câu được chuẩn hoá trước khi đọc (nguyên văn → đọc thành), 15 câu đầu:')
for u in changed[:15]:
    print(f"[{u['i']} | tr.{u['page']}] {u['text'][:80]!r}\n{'':>12}→ {u['say'][:80]!r}")

with open(ROOT/'units.jsonl', 'w', encoding='utf-8') as f:
    for u in units:
        f.write(json.dumps(u, ensure_ascii=False) + '\n')
print('\nĐã lưu', ROOT/'units.jsonl')

**Chỉ chạy ô dưới nếu ô trên cảnh báo PDF là bản scan**, rồi quay lại đặt `OCR = True` và chạy lại ô trên.

OCR tiếng Việt vẫn nhận sai dấu (vd "rằng" thành "răng"), và TTS sẽ đọc đúng cái sai đó. Nếu có bản PDF có text layer thì luôn nên dùng bản đó.

In [ ]:
# !apt-get -qq install -y tesseract-ocr tesseract-ocr-vie > /dev/null
# print('tesseract sẵn sàng')

## 4b. Sắc thái giọng đọc theo ngữ cảnh

**Ô 1 (tùy chọn): upload clip sắc thái.** Đặt `UPLOAD_REGISTERS = True`, chọn một hoặc nhiều clip, **đặt tên file đúng tên sắc thái**: `doi_thoai.wav`, `tinh_cam.wav`, `kich_tinh.wav`, `vui_tuoi.wav`, `goi_cam.wav`. Clip mẫu ở mục 2b đóng vai `trung_tinh` (kể chuyện). Mỗi clip dài 6–10 giây, cùng một người đọc, cùng mic. Không upload clip nào cũng được: khi đó chỉ tốc độ và khoảng nghỉ thay đổi theo sắc thái.

**Ô 2: gán sắc thái.** Chọn cách gán bằng `TAGGER`:
- **`'local'` (mặc định, miễn phí):** một LLM mở chạy ngay trên GPU T4 của Colab **chấm điểm từng đoạn văn**, có kèm 3 đoạn trước làm ngữ cảnh. Model chỉ trả lời một chữ số cho cả đoạn (1 = kể chuyện, 2–5 = cảm xúc), và notebook đọc thẳng xác suất của từng lựa chọn. Cả đoạn chỉ được đọc bằng giọng cảm xúc khi xác suất vượt **`NGUONG`**.
  - Ô in bảng "ngưỡng → bao nhiêu % câu có sắc thái". Chọn ngưỡng sao cho phần lớn sách vẫn là kể chuyện (thường 5–20%), rồi **chỉ chạy lại ô này**: điểm đã lưu trên Drive nên không phải chạy lại model.
  - Lời thoại (ngoặc kép, gạch đầu dòng) do luật quyết định, không hỏi LLM.
  - Muốn nhãn tinh hơn, dùng model lớn hơn ở dạng 4-bit: `LOCAL_MODEL = 'Qwen/Qwen3-8B'` hoặc `'Qwen/Qwen3-14B'`, kèm `LOCAL_4BIT = True`. Chạy chậm hơn, và nhớ đặt `RESCORE = True`.
- **`'claude'` (tốn phí):** Claude gán nhãn từng câu. Cần secret **`ANTHROPIC_API_KEY`** (biểu tượng 🔑 ở thanh bên trái → *Add new secret* → bật *Notebook access*). Mặc định gửi qua Batch API (rẻ hơn 50%, phải chờ). `TAG_MODEL = 'claude-haiku-5-5'` rẻ hơn Opus khoảng 40 lần. Cuối ô in số tiền thật.
- **`'heuristic'`:** không dùng LLM. Câu là lời thoại thì đọc kiểu `doi_thoai`, còn lại đọc `trung_tinh`.

Sửa tay câu nào thì dùng `OVERRIDES` (số thứ tự câu in ở mục 4). Câu kể chuyện bình thường giữ nguyên khóa cache cũ, chỉ câu đổi sắc thái mới phải render lại.

In [ ]:
import re, shutil
from collections import Counter
from pathlib import Path
import sac_thai

UPLOAD_REGISTERS = False   # True = upload thêm clip sắc thái (tên file = tên sắc thái, vd tinh_cam.wav)

AUDIO_EXT = ('.wav', '.mp3', '.m4a', '.flac', '.ogg')
REG_DIR = ROOT/'registers'
REG_DIR.mkdir(exist_ok=True)
if UPLOAD_REGISTERS:
    up = files.upload()
    for n in up:
        stem = re.sub(r'\s*\(\d+\)$', '', Path(n).stem)   # Colab đổi 'x.wav' trùng tên thành 'x (1).wav'
        ext = Path(n).suffix.lower()
        if stem not in sac_thai.REGISTERS or stem == sac_thai.DEFAULT or ext not in AUDIO_EXT:
            print(f'⏭  bỏ qua {n}: tên file phải là một trong',
                  ', '.join(r for r in sac_thai.REGISTERS if r != sac_thai.DEFAULT))
            continue
        for old in REG_DIR.glob(stem + '.*'):
            old.unlink()
        shutil.copy(n, REG_DIR/(stem + ext))
        print('Đã lưu clip sắc thái:', stem + ext)

REG_CLIPS = {p.stem: str(p) for p in sorted(REG_DIR.iterdir())
             if p.stem in sac_thai.REGISTERS and p.suffix.lower() in AUDIO_EXT}
REG_CLIPS[sac_thai.DEFAULT] = REF_AUDIO
print(f"{'sắc thái':<11} {'clip dùng':<11} speed  nghỉ   mô tả")
for name, spec in sac_thai.REGISTERS.items():
    voice = sac_thai.resolve(name, REG_CLIPS)
    print(f"{name:<11} {voice + ('' if voice == name else ' ↩'):<11} {spec['speed']:<5}  {spec['pause']:<5}  {spec['label']}")
print('↩ = chưa có clip riêng, dùng clip dự phòng')

In [ ]:
import json
from collections import Counter

TAGGER = 'local'               # 'local' = LLM mở trên GPU Colab (miễn phí) | 'claude' = Claude API (tốn phí) | 'heuristic'
NGUONG = 0.6                   # local: xác suất tối thiểu để đọc cả đoạn bằng giọng cảm xúc (cao = ít cảm xúc hơn)
LOCAL_MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
LOCAL_4BIT = False             # True cho model 8B/14B (vd 'Qwen/Qwen3-8B') để vừa VRAM T4
LOCAL_BATCH = 8                # số đoạn chấm cùng lúc; hết VRAM thì giảm xuống 4 hoặc 2
RESCORE = False                # local: True = chấm lại từ đầu (vd sau khi đổi LOCAL_MODEL)
TAG_MODEL = 'claude-opus-5-5'  # chỉ dùng khi TAGGER = 'claude'; 'claude-haiku-5-5' rẻ hơn khoảng 40 lần
TAG_EFFORT = 'low'
USE_BATCH = True               # Claude: True = Batch API, rẻ hơn 50% nhưng phải chờ
RETAG = False                  # Claude: True = bỏ nhãn đã lưu, gán lại từ đầu
OVERRIDES = {}                 # sửa tay: {số thứ tự câu: 'tinh_cam', ...}

assert all('say' in u for u in units), 'Hãy chạy lại mục 4 (danh sách câu) trước.'

if TAGGER == 'local':
    SCORES = ROOT/'scores_local.json'
    scores = {} if RESCORE or not SCORES.exists() else json.loads(SCORES.read_text(encoding='utf-8'))
    paras = sac_thai.paragraphs(units)
    if any(len(t) >= 40 and sac_thai._h(t) not in scores for t in paras):
        import torch
        assert torch.cuda.is_available(), 'TAGGER = "local" cần GPU (Runtime → Change runtime type → T4 GPU)'
        if LOCAL_4BIT:
            import subprocess, sys
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes'], check=True)
        scorer = sac_thai.LocalScorer(LOCAL_MODEL, batch=LOCAL_BATCH, load_4bit=LOCAL_4BIT)
        scores = scorer.score(paras, cache=scores, save=lambda c: SCORES.write_text(
            json.dumps(c), encoding='utf-8'))
        scorer.close()             # trả VRAM cho OmniVoice ở mục 6
        del scorer
    print('\nNgưỡng → số câu đọc bằng giọng cảm xúc:')
    for th, n, pct in sac_thai.threshold_table(units, scores):
        print(f"  NGUONG = {th:.1f}: {n:>5} câu ({pct:4.1f}%)" + ('   ← đang dùng' if abs(th - NGUONG) < 1e-9 else ''))
    sac_thai.assign_scored(units, set(REG_CLIPS), scores, NGUONG, overrides=OVERRIDES)
else:
    TAGS = ROOT/'tags_claude.json'
    tag_cache = {} if RETAG or not TAGS.exists() else json.loads(TAGS.read_text(encoding='utf-8'))
    save_tags = lambda c: TAGS.write_text(json.dumps(c, ensure_ascii=False), encoding='utf-8')
    tagger = None
    todo = [u for u in units if u['i'] not in OVERRIDES and u['hash'] not in tag_cache]
    if TAGGER == 'claude' and todo:
        from google.colab import userdata
        try:
            api_key = userdata.get('ANTHROPIC_API_KEY')
        except Exception:
            api_key = None
        if api_key:
            tagger = sac_thai.ClaudeTagger(api_key, model=TAG_MODEL, effort=TAG_EFFORT, use_batch=USE_BATCH,
                                           state_path=ROOT/'tag_batch.json')
        else:
            print('⚠️ Chưa có secret ANTHROPIC_API_KEY (hoặc chưa bật Notebook access) → dùng heuristic.')
    tag_cache = sac_thai.assign(units, set(REG_CLIPS), cache=tag_cache, overrides=OVERRIDES,
                                tagger=tagger, save=save_tags)
    save_tags(tag_cache)
    if tagger and tagger.usage.requests:
        print('Chi phí lượt này:', tagger.usage)

cnt = Counter(u['reg'] for u in units)
print('\nSố câu theo sắc thái:', dict(cnt.most_common()))
for name in sac_thai.REGISTERS:
    rows = [u for u in units if u['reg'] == name][:3]
    if rows and name != sac_thai.DEFAULT:
        print(f'\n--- {name} (đọc bằng clip {rows[0]["voice"]}, speed {rows[0]["speed"]})')
        for u in rows:
            print(f"[{u['i']} | tr.{u['page']}] {u['text'][:150]}")
print('\nChỗ nào chưa hợp lý: chỉnh NGUONG hoặc thêm vào OVERRIDES rồi chạy lại ô này (không chạy lại model).')


## 5. Cấu hình giọng clone

Điền **đúng lời trong clip** vào `REF_TEXT`. Để trống thì model nạp thêm Whisper để tự chép, mà chép sai một chữ là clone kém đi. Clip sắc thái ở mục 4b thì điền vào `REG_TEXTS`.

In [ ]:
REF_TEXT = ''        # lời ĐÚNG trong clip, vd: 'Xin chào, đây là giọng đọc thử của tôi.'
REG_TEXTS = {}       # lời đúng của từng clip sắc thái (mục 4b), vd {'tinh_cam': 'Mẹ ơi, con về rồi đây...'}
NUM_STEP = 32        # 16 = nhanh ~2x, 32 = nét hơn. Nên nghe thử cả hai trước khi render cả cuốn.
VOICE_ID = 'clone1'  # ĐỔI tên khi đổi clip giọng / REF_TEXT, để cache không bị lẫn
SHARD_SIZE = 300     # cứ 300 câu lưu 1 đợt lên Drive (runtime chết thì mất tối đa chừng này)
CPS_MIN, CPS_MAX = 6, 50   # ngưỡng QC (ký tự/giây); chỉnh lại theo số liệu benchmark

## 6. Nạp model + tạo prompt giọng clone cho từng clip sắc thái (làm 1 lần)

In [ ]:
import torch, numpy as np, soundfile as sf
from omnivoice import OmniVoice

model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map='cuda', dtype=torch.float16)

texts = {**REG_TEXTS, sac_thai.DEFAULT: REF_TEXT}
PROMPTS = {}
for name in sorted({u['voice'] for u in units} | {sac_thai.DEFAULT}):   # chỉ những clip thực sự được dùng
    kw = {'ref_audio': REG_CLIPS[name]}
    if texts.get(name, '').strip():
        kw['ref_text'] = texts[name].strip()
    PROMPTS[name] = model.create_voice_clone_prompt(**kw)
print('Đã có giọng clone:', ', '.join(PROMPTS))

## 7. Render

**Lần đầu để `LIMIT = 30`**: chỉ render 30 câu để đo tốc độ và ước tính tổng thời gian. Nghe thử vài file trong `cache/`. Nếu ETA hợp lý thì đặt `LIMIT = None` và chạy tiếp.

- Cứ `SHARD_SIZE` câu lại lưu một shard `.tar` lên Drive. Runtime chết thì mount lại, điền `BOOK`, chạy lại từ mục 2 đến mục 7, nó tự khôi phục và render tiếp phần thiếu.
- Mỗi câu được kiểm tra (không im lặng, không quá ngắn, tốc độ nói hợp lý). Câu lỗi thử lại 1 lần, vẫn lỗi thì ghi vào `failed.jsonl` và **không** vào cache.
- Giữ nguyên thứ tự sách, để các chương đầu hoàn tất trước.

In [ ]:
import json, os, shutil, tarfile, tempfile, time
from pathlib import Path

LIMIT = 30            # benchmark: render 30 câu rồi dừng. Đặt None để render hết.
RETRY_FAILED = False  # True = thử lại cả những câu bị loại ở lần chạy trước

OUT = Path('cache')/f'{VOICE_ID}-s{NUM_STEP}'      # tách theo giọng/step, không bao giờ lẫn
OUT.mkdir(parents=True, exist_ok=True)
SHARDS = ROOT/f'{VOICE_ID}-s{NUM_STEP}'
SHARDS.mkdir(exist_ok=True)
FAILED, DONE = SHARDS/'failed.jsonl', SHARDS/'done.txt'
SR = getattr(model, 'sampling_rate', 24000)

# 1) Phiên mới (đĩa trống): khôi phục những gì đã render từ Drive
if not any(OUT.glob('*.wav')):
    shards = sorted(SHARDS.glob('shard_*.tar'))
    for tp in shards:
        with tarfile.open(tp) as tf:
            try:
                tf.extractall(OUT, filter='data')
            except TypeError:
                tf.extractall(OUT)
    print(f'Khôi phục {len(shards)} shard từ Drive → {len(list(OUT.glob("*.wav")))} file')

# 2) File có trên đĩa nhưng chưa nằm trong shard nào (kernel chết giữa chừng) → đưa vào hàng đợi lưu
done = set(DONE.read_text().split()) if DONE.exists() else set()
buf = sorted({p.stem for p in OUT.glob('*.wav')} - done)

def flush():
    """Gói các file mới thành 1 shard rồi đẩy lên Drive (ghi .part rồi đổi tên để không bao giờ có shard dở)."""
    global buf
    if not buf:
        return
    name = f'shard_{len(list(SHARDS.glob("shard_*.tar"))):04d}.tar'
    tmp = Path(tempfile.gettempdir())/name
    with tarfile.open(tmp, 'w') as tf:
        for h in buf:
            tf.add(OUT/f'{h}.wav', arcname=f'{h}.wav')
    part = SHARDS/(name + '.part')
    shutil.copy(tmp, part)
    os.replace(part, SHARDS/name)
    tmp.unlink()
    with open(DONE, 'a') as f:
        f.write('\n'.join(buf) + '\n')
    print(f'  💾 đã lưu {name} ({len(buf)} file) lên Drive')
    buf = []

def to_np(a):
    if hasattr(a, 'detach'):
        a = a.detach().cpu().float().numpy()
    return np.squeeze(np.asarray(a, dtype=np.float32))

def qc(a, text):
    dur = len(a) / SR
    if dur < 0.3:
        return 'too_short', 0
    if float(np.max(np.abs(a))) < 0.01:
        return 'silent', 0
    cps = len(text) / dur
    if cps > CPS_MAX:
        return 'too_fast (cụt?)', cps
    if cps < CPS_MIN:
        return 'too_slow (dài/ảo?)', cps
    return 'ok', cps

# 3) Hàng đợi: bỏ trùng (giữ thứ tự sách), bỏ câu đã có, bỏ câu từng bị loại
failed = set()
if FAILED.exists():
    failed = {json.loads(l)['hash'] for l in open(FAILED, encoding='utf-8') if l.strip()}
uniq = {}
for u in units:
    uniq.setdefault(u['key'], (u['say'], u['voice'], u['speed']))
have = {p.stem for p in OUT.glob('*.wav')}
pending = [(h, *v) for h, v in uniq.items() if h not in have and (RETRY_FAILED or h not in failed)]
print(f'{len(units)} unit → {len(uniq)} câu khác nhau | đã có {len(have & set(uniq))} | '
      f'cần render {len(pending)} | từng bị loại {len(failed)}')

todo = pending[:LIMIT] if LIMIT else pending
ok = bad = 0
cps_list = []
t0 = time.time()
try:
    for i, (h, t, voice, speed) in enumerate(todo, 1):
        status, cps, a = 'error', 0, None
        for attempt in range(2):
            try:
                extra = {} if speed == 1.0 else {'speed': speed}
                a = to_np(model.generate(text=t, voice_clone_prompt=PROMPTS[voice], num_step=NUM_STEP, **extra)[0])
                status, cps = qc(a, t)
                if status == 'ok':
                    break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                status = 'oom'
            except Exception as e:
                status = 'error: ' + str(e)[:100]
        if status == 'ok':
            sf.write(str(OUT/f'{h}.wav'), a, SR)
            buf.append(h)
            ok += 1
            cps_list.append(cps)
        else:
            bad += 1
            with open(FAILED, 'a', encoding='utf-8') as f:
                f.write(json.dumps({'hash': h, 'text': t, 'voice': voice, 'speed': speed, 'why': status},
                                   ensure_ascii=False) + '\n')
        if len(buf) >= SHARD_SIZE:
            flush()
        if i % 20 == 0 or i == len(todo):
            sp = i / (time.time() - t0)
            print(f'{i}/{len(todo)}  ok={ok} loại={bad}  ~{sp:.2f} câu/s  còn ~{(len(todo) - i) / sp / 3600:.1f} giờ')
finally:
    flush()   # kể cả khi bạn bấm Stop, phần đã render vẫn được lưu

elapsed = time.time() - t0
if cps_list:
    p = np.percentile(cps_list, [1, 5, 50, 95, 99])
    print('\nKý tự/giây của câu đạt (p1, p5, p50, p95, p99):', np.round(p, 1),
          '→ chỉnh CPS_MIN / CPS_MAX cho khớp rồi mới render cả cuốn')
if LIMIT and todo:
    rate = sum(len(t) for _, t, *_ in todo) / elapsed                      # ký tự/giây xử lý
    eta_h = sum(len(t) for _, t, *_ in pending) / rate / 3600
    print(f'\n⏱ Benchmark: ước tính render hết {len(pending)} câu cần ~{eta_h:.1f} giờ GPU '
          f'(Colab free thường ngắt trước ~10-12 giờ, hay sớm hơn).')
    print('   Quá lâu? Giảm NUM_STEP, hoặc cân nhắc GPU thuê theo giờ thay vì Colab free.')
elif not pending:
    print('\nKhông còn câu nào cần render.')
else:
    print('\nXONG lượt render.')

## 8. Báo cáo theo chương

Cho biết mỗi chương đã render được bao nhiêu câu. Câu nào bị loại (`failed.jsonl`) hoặc chưa render sẽ bị **bỏ qua** khi ghép mp3, nên chương nào chưa 100% sẽ có khoảng trống trong nội dung.

In [ ]:
have = {p.stem for p in OUT.glob('*.wav')}
tot, dn = Counter(), Counter()
for u in units:
    tot[u['ch']] += 1
    dn[u['ch']] += u['key'] in have

print('   chương  đã render     %')
for c in res['chapters']:
    n, d = tot[c['id']], dn[c['id']]
    mark = '✅' if n and d == n else '  '
    print(f"{mark} {c['id']:>3}   {d:>5}/{n:<5} {100 * d / max(n, 1):5.1f}%  {c['title'][:50]}")
regs = Counter((u['reg'], u['key'] in have) for u in units)
print('\nTheo sắc thái (đã render/tổng):',
      ', '.join(f"{r} {regs[(r, True)]}/{regs[(r, True)] + regs[(r, False)]}" for r in sac_thai.REGISTERS
                if regs[(r, True)] + regs[(r, False)]))
if FAILED.exists():
    print('\nCâu bị loại (xem lại / render bù bằng RETRY_FAILED=True):', FAILED)

## 9. Ghép thành mp3

Cần đã render xong (mục 7). Nếu runtime vừa khởi động lại, chạy lại mục 2 → 7 trước, mục 7 sẽ tự khôi phục từ Drive và báo không còn câu cần render.

- `MODE = 'chapter'`: mỗi chương một file, kèm thẻ ID3 (tên chương, số thứ tự) nên app nghe sách tự xếp đúng thứ tự. `MODE = 'book'`: cả cuốn một file.
- Mỗi câu được cân âm lượng về cùng mức, rồi nghỉ ngắn giữa câu, nghỉ lâu hơn khi hết đoạn.
- Chỉ cần ghép lại (đổi `MODE`, bitrate, độ nghỉ) thì **không phải render lại**.
- Chương đã ghép và không thay đổi (cùng các câu, cùng sắc thái, cùng độ nghỉ) sẽ được bỏ qua ở lần chạy sau.
- Khoảng nghỉ được nhân theo sắc thái của câu (vd `tinh_cam` nghỉ lâu hơn), chỉnh ở `sac_thai.REGISTERS` (xem mục 3).
- Dung lượng: khoảng **29 MB mỗi giờ audio** ở 64 kbps.

In [ ]:
import hashlib, json, os, re, shutil, subprocess, tempfile, unicodedata
import numpy as np, soundfile as sf
from pathlib import Path

MODE = 'chapter'        # 'chapter' = mỗi chương 1 file mp3 | 'book' = cả cuốn 1 file
BOOK_TITLE = ''         # tên hiển thị trong app nghe sách (trống = dùng tên thư mục)
MP3_KBPS = 64           # mono 64 kbps ≈ 29 MB/giờ; 48 nhỏ hơn, 96 nét hơn
PAUSE_SENT, PAUSE_CLAUSE, PAUSE_PARA = 0.35, 0.15, 0.8   # giây nghỉ: hết câu | hết mệnh đề bị cắt | hết đoạn
ALLOW_MISSING = True    # True = chương thiếu câu vẫn ghép (bỏ câu thiếu, liệt kê trong missing.txt); False = bỏ qua cả chương
REBUILD_MP3 = False     # True = ghép lại tất cả dù file đã có

assert shutil.which('ffmpeg'), 'Thiếu ffmpeg: chạy  !apt-get -qq install -y ffmpeg'
assert any(OUT.glob('*.wav')), 'Chưa có file render nào. Chạy mục 7 trước.'

MP3_DIR = ROOT/'mp3'
MP3_DIR.mkdir(exist_ok=True)
BUILT = MP3_DIR/'built.json'
built = json.loads(BUILT.read_text(encoding='utf-8')) if BUILT.exists() else {}
_END = re.compile(r"[.!?…][\"”’')\]]*$")

def fslug(s):
    s = s.replace('đ', 'd').replace('Đ', 'D')
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-zA-Z0-9]+', '_', s).strip('_').lower()[:40] or 'untitled'

def norm(a, target_db=-20.0):
    # đưa mỗi câu về cùng mức âm lượng (giới hạn ±6 dB, không để méo)
    rms = float(np.sqrt(np.mean(a ** 2))) + 1e-9
    a = a * float(np.clip(10 ** ((target_db - 20 * np.log10(rms)) / 20), 0.5, 2.0))
    peak = float(np.max(np.abs(a)))
    return a * min(1.0, 0.98 / peak) if peak > 0 else a

def pcm(a):
    return (np.clip(a, -1, 1) * 32767).astype('<i2').tobytes()

def gap_after(u, nxt):
    if nxt is None:
        return 0.0
    scale = sac_thai.REGISTERS[u.get('reg', sac_thai.DEFAULT)]['pause']   # vd tình cảm nghỉ lâu hơn
    if nxt.get('p') != u.get('p'):
        return PAUSE_PARA * scale
    return (PAUSE_SENT if _END.search(u['text']) else PAUSE_CLAUSE) * scale

def encode(group, title, track, total, path):
    cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-f', 's16le', '-ar', str(SR), '-ac', '1', '-i', 'pipe:0',
           '-codec:a', 'libmp3lame', '-b:a', f'{MP3_KBPS}k', '-id3v2_version', '3',
           '-metadata', f'title={title}', '-metadata', f'album={BOOK_TITLE or BOOK}', '-metadata', f'track={track}/{total}',
           str(path)]
    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE)
    secs, missing = 0.0, []
    try:
        for k, u in enumerate(group):
            f = OUT/f"{u['key']}.wav"
            if not f.exists():
                missing.append(u)
                continue
            a, sr = sf.read(str(f), dtype='float32')
            assert sr == SR, f'{f.name}: sample rate {sr} khác {SR}'
            if a.ndim > 1:
                a = a.mean(axis=1)
            proc.stdin.write(pcm(norm(a)))
            gap = gap_after(u, group[k + 1] if k + 1 < len(group) else None)
            if gap:
                proc.stdin.write(np.zeros(int(SR * gap), dtype='<i2').tobytes())
            secs += len(a) / SR + gap
    finally:
        proc.stdin.close()
        rc = proc.wait()
    if rc != 0:
        raise RuntimeError('ffmpeg lỗi, xem thông báo phía trên')
    return secs, missing

by_ch = {}
for u in units:
    by_ch.setdefault(u['ch'], []).append(u)
if MODE == 'book':
    groups = [(BOOK_TITLE or BOOK, units)]
else:
    groups = [(c['title'], by_ch[c['id']]) for c in res['chapters'] if c['id'] in by_ch]
w = len(str(len(groups)))

active = []   # các nhóm hiện có file mp3 (vừa ghép hoặc đã có sẵn)
for n, (title, group) in enumerate(groups, 1):
    name = f'{BOOK}.mp3' if MODE == 'book' else f'{n:0{w}d}_{fslug(title)}.mp3'
    have_keys = [u['key'] for u in group if (OUT/f"{u['key']}.wav").exists()]
    present = len(have_keys)
    sig = hashlib.sha1('|'.join(have_keys + [f'{PAUSE_SENT},{PAUSE_CLAUSE},{PAUSE_PARA},{MP3_KBPS}']).encode()).hexdigest()[:16]
    if present == 0:
        print(f'⏭  {name}: chưa có câu nào được render, bỏ qua')
        continue
    if present < len(group) and not ALLOW_MISSING:
        print(f'⏭  {name}: thiếu {len(group) - present}/{len(group)} câu (ALLOW_MISSING=False), bỏ qua')
        continue
    if not REBUILD_MP3 and (MP3_DIR/name).exists() and built.get(name) == sig:
        print(f'✔  {name}: đã có, không đổi')
        active.append((title, group))
        continue
    tmp = Path(tempfile.gettempdir())/name
    secs, missing = encode(group, title, n, len(groups), tmp)
    part = MP3_DIR/(name + '.part')
    shutil.copy(tmp, part)
    os.replace(part, MP3_DIR/name)
    mb = tmp.stat().st_size / 1e6
    tmp.unlink()
    built[name] = sig
    BUILT.write_text(json.dumps(built, ensure_ascii=False, indent=1), encoding='utf-8')
    h, m, s_ = int(secs // 3600), int(secs % 3600 // 60), int(secs % 60)
    note = f'  ⚠️ thiếu {len(missing)} câu' if missing else ''
    print(f'🎧 {name}: {h}:{m:02d}:{s_:02d} | {mb:.1f} MB{note}')
    active.append((title, group))

missing_lines = [f"[{t[:40]} | tr.{u['page']}] {u['text']}"
                 for t, g in active for u in g if not (OUT/f"{u['key']}.wav").exists()]
MISSING = MP3_DIR/'missing.txt'
if missing_lines:
    MISSING.write_text('\n'.join(missing_lines), encoding='utf-8')
    print(f'\n⚠️ {len(missing_lines)} câu bị bỏ qua trong các file mp3 hiện có, danh sách: {MISSING}')
elif MISSING.exists():
    MISSING.unlink()
print(f'\nXong. Tải về từ Drive: omnivoice_render/{BOOK}/mp3/  (chọn thư mục → chuột phải → Download)')

## 10. Tải về

File mp3 nằm trên Drive tại `omnivoice_render/<sách>/mp3/`. Mở drive.google.com, chọn thư mục `mp3` → **Download** (Drive tự nén và tải về). Với file hàng trăm MB, cách này ổn định hơn nhiều so với tải thẳng từ Colab.

Muốn nghe thử nhanh một chương ngắn ngay trong Colab, bỏ dấu `#` ở dòng dưới:

In [ ]:
# files.download(str(sorted(MP3_DIR.glob('*.mp3'))[0]))